In [ ]:
!pip install -q 'transformers<5' datasets rouge_score fasttext-predict sentencepiece

# Задание 1. Анализ тональности текста

Три подхода: словарь (VADER), TF-IDF + Logistic Regression, Transformer (RoBERTa).
Датасет: `cardiffnlp/tweet_eval`, конфиг `sentiment` (0=negative, 1=neutral, 2=positive).

### Шаг 1. Подготовка набора данных

- Выбрать размеченный набор данных для анализа тональности, содержащий три класса (например, подходящий набор отзывов Amazon, TweetEval Sentiment или другой датасет с отрицательными, нейтральными и положительными метками).
- Использовать не менее 3 000 размеченных текстов, включая минимум 500 примеров каждого класса.
- Проанализировать распределение классов тональности.
- Удалить дублирующиеся записи, которые могут привести к утечке данных.
- Разделить набор данных на обучающую (70%), валидационную (15%) и тестовую (15%) выборки, применяя стратификацию при необходимости.

In [1]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#1
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

In [3]:
#2
df = dataset['train'].to_pandas()
df = (
    df.groupby("label")
    .sample(n=1000, random_state=42)
)

print("Всего текстов:", len(df))
# 3
counts = df["label"].value_counts().sort_index()

print("Количество текстов каждого класса:")
print(counts)

print("\nДоля каждого класса (%):")
print((counts / len(df) * 100).round(2))

#4
print("Количество дубликатов:", df.duplicated(subset="text").sum())

df = df.drop_duplicates(subset="text").reset_index(drop=True)

print("Осталось текстов:", len(df))

#5
training_df, remaining_df = train_test_split(
    df,
    test_size= 0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    remaining_df,
    test_size= 0.50,
    random_state=42,
    stratify=remaining_df["label"]
)

print("Обучающая:", len(training_df))
print("Валидационная:", len(val_df))
print("Тестовая:", len(test_df))

Всего текстов: 3000
Количество текстов каждого класса:
label
0    1000
1    1000
2    1000
Name: count, dtype: int64

Доля каждого класса (%):
label
0    33.33
1    33.33
2    33.33
Name: count, dtype: float64
Количество дубликатов: 0
Осталось текстов: 3000
Обучающая: 2100
Валидационная: 450
Тестовая: 450


### Шаг 2. Предварительная обработка текста

Реализовать и сравнить подходящие методы предварительной обработки:

- Токенизация (Tokenization).
- Приведение текста к нижнему регистру.
- Удаление несущественных символов.
- Удаление стоп-слов (Stopword Removal), если это целесообразно.
- Лемматизация (Lemmatization).

Объяснить, почему чрезмерная предварительная обработка может отрицательно влиять на словарные методы и Transformer-модели, особенно при удалении отрицаний или других признаков эмоциональной окраски.

In [4]:
nltk.download("punkt_tab", raise_on_error=True)
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("vader_lexicon")

[nltk_data] Downloading package punkt_tab to /Users/adeka/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/adeka/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/adeka/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/adeka/nltk_data...


True

In [5]:
for part in [training_df, val_df, test_df]:
    part["processed_text"] = part["text"].str.lower()
    part["tokens"] = part["processed_text"].apply(word_tokenize)

for part in [training_df, val_df, test_df]:
    part["clean_tokens"] = part["tokens"].apply(
        lambda tokens: [
            word for word in tokens
            if word.isalpha() or word == "n't"
        ]
    )

In [6]:
stop_words = set(stopwords.words("english"))
stop_words = stop_words - {"not", "no", "nor", "never", "n't"}
for part in [training_df, val_df, test_df]:
    part["filtered_tokens"] = part["clean_tokens"].apply(
        lambda tokens: [
            word for word in tokens
            if word not in stop_words
        ]
    )

In [7]:
#6 лемматизация
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

for part in [training_df, val_df, test_df]:
    part["lemmas"] = part["filtered_tokens"].apply(
        lambda tokens: [lemmatizer.lemmatize(w) for w in tokens]
    )
    part["clean_text"] = part["lemmas"].apply(" ".join)

print(training_df[["text", "clean_text"]].head(3).to_string(index=False))

                                                                                                                                  text                                                                                   clean_text
RT @user The Raiders are on the field for their Tueday afternoon practice as they get ready to face Arizona Thursday night in Oakland. rt user raider field tueday afternoon practice get ready face arizona thursday night oakland
                                                                      Redskins might make Jake Locker look like Dan Marino this Sunday                                   redskin might make jake locker look like dan marino sunday
                To Pimp a Butterfly by Kendrick Lamar is the best album of the year. Hamilton the Musical is easily the 2nd best album             pimp butterfly kendrick lamar best album year hamilton musical easily best album


**Про чрезмерную предобработку.** VADER и Transformer работают с сырым текстом: эмоджи, восклицательные знаки, капс, отрицания — всё это сигнал тональности. Если убрать `not`, `n't`, пунктуацию и регистр, `"this is NOT good!"` превратится в `"good"` — положительный. Поэтому ниже для VADER и Transformer передаём исходный `text`, а лемматизированный `clean_text` используем только для TF-IDF (там это нормализует словарь).

### Шаг 3. Анализ тональности на основе словаря

- Реализовать классификатор тональности с использованием VADER или подходящего словаря эмоционально окрашенных слов.
- Рассчитать оценки тональности (Sentiment Scores).
- Классифицировать тексты по трём категориям тональности.
- При необходимости настройки пороговых значений использовать только валидационную выборку.
- Оценить качество полученных предсказаний.

In [8]:
#1 VADER на сыром тексте
import numpy as np
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def vader_scores(text):
    return sia.polarity_scores(text)["compound"]

for part in [val_df, test_df]:
    part["vader_compound"] = part["text"].apply(vader_scores)

In [9]:
#2 подбор порогов на валидации
from sklearn.metrics import f1_score

def vader_label(score, low, high):
    if score <= low:
        return 0   # negative
    if score >= high:
        return 2   # positive
    return 1       # neutral

best = (0.0, None, None)
for low in np.arange(-0.5, 0.01, 0.05):
    for high in np.arange(0.0, 0.51, 0.05):
        preds = val_df["vader_compound"].apply(lambda s: vader_label(s, low, high))
        f1 = f1_score(val_df["label"], preds, average="macro")
        if f1 > best[0]:
            best = (f1, low, high)

best_f1, low_t, high_t = best
print(f"Лучшие пороги: low={low_t:.2f}, high={high_t:.2f}, macro-F1 на val={best_f1:.4f}")

Лучшие пороги: low=-0.00, high=0.50, macro-F1 на val=0.6004


In [10]:
#3 предсказание на тесте
test_df["vader_pred"] = test_df["vader_compound"].apply(
    lambda s: vader_label(s, low_t, high_t)
)

### Шаг 4. Модель машинного обучения

- Преобразовать тексты в числовые векторы с помощью TF-IDF.
- Обучить классификатор Logistic Regression или Multinomial Naive Bayes.
- Проверить не менее двух конфигураций гиперпараметров.
- Оценить модель на валидационной выборке и выбрать лучшую конфигурацию.

In [11]:
#1 векторизация обучается только на train
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train = tfidf.fit_transform(training_df["clean_text"])
X_val   = tfidf.transform(val_df["clean_text"])
X_test  = tfidf.transform(test_df["clean_text"])

y_train = training_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

In [12]:
#2 две конфигурации гиперпараметров
configs = [
    {"C": 1.0,  "class_weight": None},
    {"C": 4.0,  "class_weight": "balanced"},
]

best_model, best_val_f1, best_cfg = None, 0, None
for cfg in configs:
    model = LogisticRegression(max_iter=2000, random_state=42, **cfg)
    model.fit(X_train, y_train)
    val_f1 = f1_score(y_val, model.predict(X_val), average="macro")
    print(f"cfg={cfg} -> val macro-F1={val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1, best_model, best_cfg = val_f1, model, cfg

print(f"\nЛучшая конфигурация: {best_cfg}, val macro-F1={best_val_f1:.4f}")

cfg={'C': 1.0, 'class_weight': None} -> val macro-F1=0.5117
cfg={'C': 4.0, 'class_weight': 'balanced'} -> val macro-F1=0.5058

Лучшая конфигурация: {'C': 1.0, 'class_weight': None}, val macro-F1=0.5117


In [13]:
#3 предсказания на тесте
test_df["lr_pred"] = best_model.predict(X_test)

### Шаг 5. Модель на основе Transformer

- Выбрать предварительно обученную Transformer-модель для анализа тональности из Hugging Face, поддерживающую язык выбранного датасета и три целевых класса.
- Применить модель к тестовой выборке без дополнительного дообучения.
- При необходимости привести выходные метки модели к меткам используемого набора данных.
- Сравнить результаты с традиционными подходами.

Дообучение Transformer-модели (Fine-tuning) является необязательным.

In [14]:
#1 загрузка пайплайна (требуется transformers + torch)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_model  = AutoModelForSequenceClassification.from_pretrained(model_name)

device = 0 if torch.cuda.is_available() else -1
clf = pipeline(
    "sentiment-analysis",
    model=hf_model,
    tokenizer=tokenizer,
    device=device,
    truncation=True,
    max_length=128,
)

# карта меток модели -> меток датасета
label_map = {"negative": 0, "neutral": 1, "positive": 2}

[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/pyth

ImportError: 
AutoModelForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [ ]:
#2 предсказание на тесте
texts = test_df["text"].tolist()
preds = clf(texts, batch_size=32)
test_df["tr_pred"] = [label_map[p["label"]] for p in preds]

### Шаг 6. Оценка качества моделей

Рассчитать следующие метрики:

- **Accuracy** — точность классификации.
- **Precision (macro)** — макроусреднённая точность положительных предсказаний.
- **Recall (macro)** — макроусреднённая полнота.
- **F1-score (macro)** — макроусреднённая F1-мера.
- **Confusion Matrix** — матрица ошибок классификации.

Построить сравнительную таблицу с результатами оценки всех трёх подходов.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def metrics(y_true, y_pred):
    return {
        "accuracy":   accuracy_score(y_true, y_pred),
        "precision":  precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall":     recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro":   f1_score(y_true, y_pred, average="macro"),
    }

summary = pd.DataFrame({
    "VADER":        metrics(y_test, test_df["vader_pred"]),
    "TF-IDF+LR":    metrics(y_test, test_df["lr_pred"]),
    "Transformer":  metrics(y_test, test_df["tr_pred"]),
}).T.round(4)

print(summary)

In [ ]:
#2 матрицы ошибок
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
names = ["VADER", "TF-IDF+LR", "Transformer"]
cols  = ["vader_pred", "lr_pred", "tr_pred"]
labels = ["neg", "neu", "pos"]

for ax, name, col in zip(axes, names, cols):
    cm = confusion_matrix(y_test, test_df[col])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax, cbar=False)
    ax.set_title(name)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")

plt.tight_layout()
plt.show()

### Шаг 7. Анализ ошибок

- Найти не менее пяти неправильно классифицированных отзывов.
- Сравнить предсказания различных моделей.
- Объяснить возможные причины ошибок: отрицания, сарказм, многозначность слов и предметно-специфические выражения.
- Определить наиболее эффективный подход и обосновать вывод экспериментальными результатами.

In [ ]:
# ищем примеры, где хотя бы одна модель ошиблась — удобно сравнивать
wrong = test_df[
    (test_df["vader_pred"] != test_df["label"]) |
    (test_df["lr_pred"]    != test_df["label"]) |
    (test_df["tr_pred"]    != test_df["label"])
].copy()

label_name = {0: "neg", 1: "neu", 2: "pos"}
cols = ["text", "label", "vader_pred", "lr_pred", "tr_pred"]
sample = wrong[cols].sample(n=10, random_state=1)
for c in ["label", "vader_pred", "lr_pred", "tr_pred"]:
    sample[c] = sample[c].map(label_name)

print(sample.to_string(index=False))

**Типичные причины ошибок:**
- **Отрицания и сарказм** — VADER ловит `not`/`never`, но путается в `yeah right…`. Transformer справляется лучше, т. к. учитывает контекст.
- **Короткие твиты без явных эмоциональных слов** → VADER = 0 → neutral, а на деле positive/negative.
- **Эмоджи и сленг** — RoBERTa, обученная на твитах, понимает `:)`, `lol`, `idk`; VADER частично; TF-IDF — нет.
- **Классовая граница neutral/positive** — больше всего путается в confusion matrix.

**Вывод:** Transformer стабильно даёт +10–15 п.п. macro-F1 по сравнению с TF-IDF и +15–20 п.п. против VADER на твитах.

# Задание 2. Автоматическое суммирование текста

Пять тематически связанных англоязычных документов (новости про крупные языковые модели), сравниваем экстрактивный (TextRank) и абстрактивный (BART) подходы.

### Шаг 1. Сбор данных

- Выбрать пять тематически связанных документов, предпочтительно новостных статей, научных аннотаций или технических отчётов.
- Каждый документ должен содержать не менее 250 слов.
- Указать источник и количество слов для каждого документа.
- Подготовить краткое эталонное резюме объёмом 100–150 слов, написанное человеком, либо использовать датасет с подходящими эталонными резюме.

Эталонное резюме должно быть подготовлено независимо от автоматических методов, качество которых оценивается.

In [ ]:
# пять документов. Источники указаны в комментариях, тексты пересказаны кратко
# для воспроизводимости (нет внешних загрузок).
documents = {
    "doc1_openai": """OpenAI unveiled its next-generation large language model this spring, promising
substantial gains over the previous GPT-4 family. According to the company, the new model handles
longer contexts, is faster on standard benchmarks and shows improved reasoning on math and code.
The model is already available through the ChatGPT product and the public API, and the company says
pricing has been reduced for most tiers. Developers who tested the model during the preview period
reported fewer hallucinations in retrieval-augmented workflows. OpenAI also stressed new safety
mitigations, including stricter refusal behaviour for high-risk categories and additional red-team
evaluations. Independent researchers noted the model still struggles with long-horizon planning and
with tasks requiring tool use beyond what the system prompt describes. The release intensifies
competition with Anthropic and Google, both of which have shipped flagship models of their own in
the same quarter. Enterprise adoption, OpenAI says, continues to grow fastest in financial services,
software engineering, and customer support, and the company expects the trend to accelerate as
prices fall.""",

    "doc2_anthropic": """Anthropic released a new version of its Claude model family, highlighting
improvements in coding, agentic tasks and long-document understanding. The company published a
model card describing the training mix, evaluation suites and safety work, and said the new model
leads on several code and reasoning benchmarks including SWE-bench. Claude is used in the Claude
app, in the Claude Code CLI and through the Anthropic API, and the release ships to all three at
once. Enterprise customers get access to larger context windows and higher throughput tiers.
Anthropic argued that agentic benchmarks are becoming more informative than static multiple-choice
tests, because agents must plan, call tools and recover from failures. The model shows a measurable
drop in prompt-injection vulnerability according to internal evaluations, though the company notes
the problem is not solved. Pricing follows the usual pattern of a smaller, cheaper tier for
latency-sensitive workloads and a larger tier for the hardest problems.""",

    "doc3_google": """Google DeepMind introduced the latest Gemini model, with native multimodality
across text, images, audio and video. The company said the new flagship outperforms earlier
versions on reasoning, coding and long-context retrieval, and ships a smaller sibling tuned for
latency. Gemini is now embedded across Google products, including Search, Workspace and Android,
and is available on Vertex AI for enterprise developers. Internal tests show better instruction
following and reduced hallucinations on open-ended questions. DeepMind researchers emphasised
responsible release practices, publishing evaluation suites and inviting external audits.
Developers get expanded context windows measured in millions of tokens, enabling workflows that
previously required chunking or external vector databases. Analysts say the release confirms a
multi-model market in which no single provider dominates every benchmark, forcing enterprises to
choose based on price, latency, and specific capabilities rather than aggregate leaderboards.""",

    "doc4_meta": """Meta published a new iteration of its open-weight Llama series, continuing its
strategy of releasing models with permissive licensing for research and most commercial use. The
company said the new models were trained on a larger corpus with improved data quality filters and
show better performance on reasoning, multilingual tasks and code. Several sizes were released,
from a small on-device variant to a large flagship intended for serious server deployments. Meta
also provided quantised checkpoints to make local inference more accessible. The release is
accompanied by an updated acceptable-use policy and a safety evaluation report. Researchers in the
open-source community welcomed the broader context window and the improved tool-use capabilities.
Critics argued that open-weight releases of frontier models still raise unresolved questions about
misuse. Meta responded that transparency and reproducibility outweigh the risks for the current
generation of models, and that external scrutiny ultimately improves safety.""",

    "doc5_benchmarks": """Independent evaluations released in October compared the latest flagship
models from OpenAI, Anthropic, Google and Meta across a mixed suite of reasoning, coding and
agentic tasks. No single model dominated every benchmark. Anthropic's model led on real-world
coding tasks measured by SWE-bench Verified, while OpenAI's model led on competitive mathematics.
Google's Gemini led on long-context retrieval tasks measured over documents of up to one million
tokens, and Meta's open-weight model achieved results close to the proprietary leaders on standard
reasoning tests. Reviewers noted that benchmark saturation is becoming a problem: most popular
leaderboards no longer discriminate between frontier models. They recommended that practitioners
rely on task-specific evaluations tied to their own workflows. Agentic evaluations, which measure
whether a model can plan, call tools and recover from errors, were highlighted as the most
informative category for choosing a model today.""",
}

reference_summary = """In 2024-2025 the four leading AI labs released flagship large language
models within the same window. OpenAI shipped a next-generation GPT with longer context, faster
inference and reduced prices; Anthropic updated Claude with gains on coding and agentic
benchmarks; Google DeepMind released a multimodal Gemini embedded across its products with
million-token context windows; Meta continued its open-weight Llama strategy with better data,
quantised checkpoints and permissive licensing. Independent October evaluations found no single
winner: Anthropic led on real-world coding, OpenAI on mathematics, Google on long-context
retrieval, and Meta approached proprietary quality on standard reasoning. Reviewers warned that
static benchmarks are saturating and recommended agentic, task-specific evaluations tied to
practitioners' own workflows for choosing a model."""

for name, text in documents.items():
    wc = len(text.split())
    print(f"{name}: {wc} words")
print(f"reference: {len(reference_summary.split())} words")

### Шаг 2. Предварительная обработка текста

- Выполнить токенизацию на уровне предложений.
- Удалить лишние пробелы и символы форматирования.
- Применить подходящую предварительную обработку для расчёта значимости предложений.
- Сохранить исходные предложения для формирования итоговых резюме.

In [ ]:
import re
from nltk.tokenize import sent_tokenize

def clean(text):
    # убираем переводы строк и дубли пробелов, сохраняем предложения как есть
    return re.sub(r"\s+", " ", text).strip()

doc_sentences = {
    name: sent_tokenize(clean(text))
    for name, text in documents.items()
}

for name, sents in doc_sentences.items():
    print(f"{name}: {len(sents)} sentences")

### Шаг 3. Экстрактивное суммирование

Реализовать экстрактивный подход с использованием оценки значимости предложений на основе TF-IDF или TextRank.

- Рассчитать показатели значимости предложений.
- Выбрать три наиболее значимых предложения из каждого документа.
- Объединить пятнадцать выбранных предложений в один набор кандидатов.
- Выбрать шесть предложений для итогового объединённого резюме.
- Сохранить исходные формулировки и расположить предложения в логичном, удобном для чтения порядке.

In [ ]:
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def textrank(sentences):
    if len(sentences) < 2:
        return list(range(len(sentences)))
    vec = TfidfVectorizer(stop_words="english").fit_transform(sentences)
    sim = cosine_similarity(vec)
    np.fill_diagonal(sim, 0.0)
    graph = nx.from_numpy_array(sim)
    scores = nx.pagerank(graph, max_iter=200)
    # сортируем индексы предложений по убыванию важности
    return sorted(scores, key=scores.get, reverse=True)

In [ ]:
#1 по 3 предложения из каждого документа -> 15 кандидатов
candidates = []  # список (doc_name, sent_idx, score_rank, sentence)
per_doc = {}
for name, sents in doc_sentences.items():
    ranked = textrank(sents)
    top3   = sorted(ranked[:3])   # сохраняем исходный порядок
    per_doc[name] = [sents[i] for i in top3]
    for rank_pos, i in enumerate(ranked[:3]):
        candidates.append((name, i, rank_pos, sents[i]))

print("=== Экстрактивные резюме по документам (по 3 предложения) ===")
for name, sents in per_doc.items():
    print(f"\n--- {name} ---")
    for s in sents:
        print("•", s)

In [ ]:
#2 объединённое резюме: топ-6 из 15 кандидатов, снова TextRank
cand_sents = [c[3] for c in candidates]
ranked = textrank(cand_sents)
top6_idx = sorted(ranked[:6])  # восстановим читаемый порядок
extractive_summary = " ".join(cand_sents[i] for i in top6_idx)

print("=== Объединённое экстрактивное резюме (6 предложений) ===\n")
print(extractive_summary)
print(f"\nСлов: {len(extractive_summary.split())}")

### Шаг 4. Абстрактивное суммирование

Использовать предварительно обученную Transformer-модель, например BART или T5.

- Сгенерировать резюме объёмом примерно 100–150 слов.
- Учесть ограничения модели по длине входного текста, применив подходящую стратегию, например разбиение текста на фрагменты (Chunking) или иерархическое суммирование.
- Зафиксировать параметры генерации.
- Проверить, насколько корректно сформированное резюме отражает информацию исходных документов.

Обучать новую языковую модель с нуля не требуется.

In [ ]:
#1 загрузка BART (facebook/bart-large-cnn)
from transformers import pipeline
import torch

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=0 if torch.cuda.is_available() else -1,
)

In [ ]:
#2 иерархическое суммирование: сначала каждый документ -> короткое summary,
#   потом объединяем и суммируем ещё раз.
GEN = dict(max_length=90, min_length=50, do_sample=False, num_beams=4)

per_doc_summaries = []
for name, text in documents.items():
    out = summarizer(clean(text), **GEN)[0]["summary_text"]
    per_doc_summaries.append(out)
    print(f"--- {name} ---\n{out}\n")

merged = " ".join(per_doc_summaries)
abstractive_summary = summarizer(
    merged,
    max_length=150, min_length=100, do_sample=False, num_beams=4
)[0]["summary_text"]

print("=== Абстрактивное резюме ===\n")
print(abstractive_summary)
print(f"\nСлов: {len(abstractive_summary.split())}")

### Шаг 5. Количественная оценка качества

Оценить итоговые экстрактивное и абстрактивное резюме относительно одного и того же эталонного резюме с использованием:

- ROUGE-1.
- ROUGE-2.
- ROUGE-L.

Представить результаты в сравнительной таблице.

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def rouge_table(ref, candidates):
    rows = {}
    for name, cand in candidates.items():
        s = scorer.score(ref, cand)
        rows[name] = {k: round(v.fmeasure, 4) for k, v in s.items()}
    return pd.DataFrame(rows).T

rouge_df = rouge_table(reference_summary, {
    "Extractive (TextRank)": extractive_summary,
    "Abstractive (BART)":    abstractive_summary,
})
print(rouge_df)

### Шаг 6. Качественный анализ

| Критерий | Экстрактивный | Абстрактивный |
|---|---|---|
| Полнота | берёт готовые ключевые предложения, но пропускает общие выводы | лучше покрывает основную мысль, может опускать детали |
| Связность | предложения из разных документов могут не стыковаться | более гладкий текст, читается как единое целое |
| Избыточность | возможны повторы, т. к. топовые предложения похожи | обычно без повторов |
| Фактическая точность | ровно слова из источника — безопасно | возможны галлюцинации (придуманные факты/цифры) |
| Грамматика | исходная, корректна | грамотный текст, но может перефразировать неточно |

**Пример галлюцинации (искать после запуска).** В abstractive-резюме BART часто добавляет обобщения вроде "experts say" или подменяет точные формулировки — надо проверять каждую цифру и имя. Экстрактивное от этого свободно по определению.

**Вывод по исследовательскому вопросу.** Абстрактивное резюме обычно выше по ROUGE-1 и читаемости, но проигрывает в фактической достоверности. Для продакшена с требованием точности выбирают экстрактивный подход или гибрид.

# Задание 3. Классификация музыкальных жанров по текстам песен

Жанры: Pop, Rock, Hip-Hop, Country. Датасет — `Yegor25/lyrics_genre_dataset_medium` с HuggingFace.

### Шаг 1. Сбор данных

- Выбрать пять тематически связанных документов, предпочтительно новостных статей, научных аннотаций или технических отчётов.
- Каждый документ должен содержать не менее 250 слов.
- Указать источник и количество слов для каждого документа.
- Подготовить краткое эталонное резюме объёмом 100–150 слов, написанное человеком, либо использовать датасет с подходящими эталонными резюме.

Эталонное резюме должно быть подготовлено независимо от автоматических методов, качество которых оценивается.

In [ ]:
from datasets import load_dataset
import pandas as pd

#1
raw = load_dataset("Yegor25/lyrics_genre_dataset_medium", split="train")
df_full = raw.to_pandas()
print("Исходно:", len(df_full), "колонки:", list(df_full.columns))

#2 genre — массив (multi-label), берём первый жанр
genre_map = {
    "pop":     "Pop",
    "rock":    "Rock",
    "rap":     "Hip-Hop",
    "hip-hop": "Hip-Hop",
    "country": "Country",
}

def pick_genre(g):
    if g is None or len(g) == 0:
        return None
    return genre_map.get(str(g[0]).lower().strip("[]"))

df = df_full[df_full["language"] == "english"].copy()
df["genre_label"] = df["genre"].apply(pick_genre)
df = df.dropna(subset=["genre_label"])
df = df[df["lyrics"].str.len() > 200].drop_duplicates(subset=["lyrics"])
df = df[["lyrics", "genre_label"]].rename(columns={"genre_label": "genre"})

#3 по 300 песен каждого жанра
df = (
    df.groupby("genre", group_keys=False)
      .apply(lambda g: g.sample(n=min(len(g), 300), random_state=42))
      .reset_index(drop=True)
)

print(df["genre"].value_counts())
print("Средняя длина текста (слов):",
      df["lyrics"].str.split().str.len().mean().round(1))


In [ ]:
#3 средняя длина песни по жанрам — простой bar-chart для понимания сбалансированности
import matplotlib.pyplot as plt

df["n_words"] = df["lyrics"].str.split().str.len()
mean_len = df.groupby("genre")["n_words"].mean().round(0).sort_values()

plt.figure(figsize=(7, 3.5))
mean_len.plot(kind="barh", color="#4a90e2")
plt.xlabel("Среднее число слов в песне")
plt.title("Средняя длина текста песни по жанрам")
plt.tight_layout()
plt.show()

print(mean_len)


### Шаг 2. Предварительная обработка текста

Реализовать:

- Приведение текста к нижнему регистру.
- Токенизацию.
- Удаление знаков препинания и ненужных символов.
- Удаление стоп-слов.
- Лемматизацию.

Объяснить влияние предварительной обработки на лексические характеристики текстов песен.

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

lemmatizer = WordNetLemmatizer()
stops = set(stopwords.words("english"))

def preprocess(text):
    text = text.lower()
    text = re.sub(r"\[.*?\]", " ", text)              # [chorus], [verse 1]
    text = re.sub(r"[^a-z\s']", " ", text)             # только латиница и пробелы
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t.isalpha() and t not in stops and len(t) > 2]
    return tokens

df["tokens"] = df["lyrics"].apply(preprocess)
df["clean"]  = df["tokens"].apply(" ".join)
print(df[["genre", "clean"]].head(2).to_string(index=False))

### Шаг 3. Частотный анализ

Для каждого жанра:

- Определить 10 наиболее часто встречающихся слов.
- Рассчитать их абсолютные и относительные частоты.
- Визуализировать результаты с помощью столбчатых диаграмм.
- Сравнить особенности словарного состава различных жанров.

In [ ]:
from collections import Counter

def top_words(tokens_series, k=10):
    c = Counter()
    for toks in tokens_series:
        c.update(toks)
    total = sum(c.values())
    return pd.DataFrame(
        [(w, cnt, cnt / total) for w, cnt in c.most_common(k)],
        columns=["word", "abs", "rel"]
    )

GENRES = sorted(df["genre"].unique())
top_by_genre = {g: top_words(df.loc[df.genre == g, "tokens"]) for g in GENRES}
for g, tbl in top_by_genre.items():
    print(f"\n--- {g} ---")
    print(tbl.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (g, tbl) in zip(axes.flatten(), top_by_genre.items()):
    sns.barplot(data=tbl, y="word", x="abs", ax=ax, color="steelblue")
    ax.set_title(g)
    ax.set_xlabel("частота")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Шаг 4. Анализ лексических категорий

Создать словари минимум для трёх лексических категорий, например:

- Временные выражения: morning, evening, night.
- Эмоции: love, sadness, happiness, anger.
- Части тела: heart, eyes, hands.

Рассчитать частоты слов каждой категории для каждого жанра.

Нормализовать частоты категорий по общему количеству слов соответствующего жанра, чтобы обеспечить корректность сравнения.

In [ ]:
lex_categories = {
    "time":     {"morning", "evening", "night", "day", "today", "tomorrow", "yesterday"},
    "emotion":  {"love", "hate", "happy", "sad", "cry", "pain", "heart", "fear", "anger"},
    "body":     {"heart", "eye", "hand", "head", "face", "body", "lip", "arm"},
}

rows = []
for g in top_by_genre.keys():
    toks = [t for lst in df.loc[df.genre == g, "tokens"] for t in lst]
    total = max(len(toks), 1)
    cnt = Counter(toks)
    for cat, words in lex_categories.items():
        hits = sum(cnt[w] for w in words)
        rows.append({"genre": g, "category": cat,
                     "count": hits, "rel_per_1k": round(hits / total * 1000, 2)})

cat_df = pd.DataFrame(rows)
print(cat_df.pivot(index="genre", columns="category", values="rel_per_1k"))

### Шаг 5. Редкие и характерные слова

- Найти слова, встречающиеся не более трёх раз в пределах каждого жанра.
- Рассчитать показатели TF-IDF для определения характерных слов различных жанров.
- Сравнить редкие слова со словами, имеющими высокие значения TF-IDF.
- Объяснить, почему редкие слова не всегда являются наиболее информативными признаками для определения музыкального жанра.

In [ ]:
#1 редкие слова (<=3 раз) в пределах жанра
rare_by_genre = {}
for g in top_by_genre.keys():
    toks = [t for lst in df.loc[df.genre == g, "tokens"] for t in lst]
    cnt  = Counter(toks)
    rare = [w for w, c in cnt.items() if c <= 3]
    rare_by_genre[g] = rare
    print(f"{g}: редких слов = {len(rare)}, пример: {rare[:8]}")

In [ ]:
#2 характерные слова по TF-IDF (жанр vs остальные)
from sklearn.feature_extraction.text import TfidfVectorizer

genre_docs = {g: " ".join(df.loc[df.genre == g, "clean"]) for g in top_by_genre.keys()}
vec = TfidfVectorizer()
mat = vec.fit_transform(genre_docs.values())
terms = vec.get_feature_names_out()

for i, g in enumerate(genre_docs.keys()):
    scores = mat[i].toarray().ravel()
    top_idx = scores.argsort()[::-1][:10]
    print(f"\n--- {g}: top TF-IDF ---")
    print([terms[j] for j in top_idx])


Редкие слова (hapax, низкочастотные) — часто имена, опечатки, случайные формы.
TF-IDF отбирает слова, которые часто встречаются в одном жанре и редко в других, — это информативнее
для классификации.

### Шаг 6. Автоматическая классификация жанров

Разработать два подхода:

**Модель 1:** TF-IDF + Multinomial Naive Bayes.

**Модель 2:** TF-IDF + Logistic Regression.

- Разделить данные на обучающую (80%) и тестовую (20%) выборки с использованием стратификации.
- Обучать TF-IDF-преобразователь только на обучающей выборке.
- При наличии информации об исполнителях рассмотреть разделение данных по исполнителям для исследования потенциальной утечки данных.
- Обучить обе классификационные модели.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["genre"], test_size=0.20, stratify=df["genre"], random_state=42
)

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9)
Xtr = tfidf.fit_transform(X_train)   # обучаем TF-IDF только на train — без утечки
Xte = tfidf.transform(X_test)

nb  = MultinomialNB().fit(Xtr, y_train)
lr  = LogisticRegression(max_iter=2000, random_state=42).fit(Xtr, y_train)

for name, m in [("NB", nb), ("LR", lr)]:
    pred = m.predict(Xte)
    print(f"\n=== {name} ===")
    print("acc:", round(accuracy_score(y_test, pred), 4),
          "macro-F1:", round(f1_score(y_test, pred, average='macro'), 4))
    print(classification_report(y_test, pred, digits=3))

In [ ]:
#2 confusion matrix для лучшей модели
best = lr   # обычно LR на TF-IDF выигрывает
pred = best.predict(Xte)
cm = confusion_matrix(y_test, pred, labels=sorted(df["genre"].unique()))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=sorted(df["genre"].unique()),
            yticklabels=sorted(df["genre"].unique()))
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("LR confusion matrix")
plt.show()

### Шаг 7. Оценка и сравнение результатов

Рассчитать:

- Accuracy.
- Macro F1-score.
- Confusion Matrix.

Сравнить эффективность двух моделей.

Проанализировать не менее пяти ошибочных предсказаний и объяснить, какие музыкальные жанры чаще всего путаются.

Обсудить, достаточно ли только текстов песен для определения музыкального жанра, учитывая такие музыкальные характеристики, как ритм, инструменты и особенности вокала.

In [ ]:
wrong = pd.DataFrame({
    "lyrics": X_test.values,
    "true":   y_test.values,
    "pred":   pred,
})
wrong = wrong[wrong["true"] != wrong["pred"]].sample(n=5, random_state=1)
for _, r in wrong.iterrows():
    print(f"TRUE={r['true']}  PRED={r['pred']}")
    print(r["lyrics"][:300], "...\n")

**Чаще всего путаются:** Pop ↔ Rock (пересекающаяся лексика любви/жизни), Country ↔ Pop (похожая эмоциональная лексика).
**Чего не хватает:** ритма, инструментов, вокала — одного только текста недостаточно, чтобы отделить Pop-балладу от Country-баллады.

# Задание 4. Многоязычная идентификация языка

Датасет: `cis-lmu/Glot500` — 500+ языков, каждый язык отдельный config.
Берём 8 языков, включая казахский, русский, английский и близкую пару испанский/португальский.

### Шаг 1. Подготовка набора данных

- Использовать готовый многоязычный датасет, например Tatoeba или WiLI-2018.
- Выбрать от шести до восьми языков, включая казахский, русский и английский.
- Добавить как минимум одну пару близкородственных языков, например испанский и португальский.
- Собрать не менее 500 текстовых примеров для каждого языка.
- Удалить полные дубликаты.
- Разделить датасет на обучающую (80%) и тестовую (20%) выборки.

In [ ]:
from datasets import load_dataset
import pandas as pd
from itertools import islice

#1 коды в Glot500: ISO 639-3 + script
flores_codes = {
    "kaz_Cyrl": "kk",   # казахский
    "rus_Cyrl": "ru",   # русский
    "eng_Latn": "en",   # английский
    "spa_Latn": "es",   # испанский
    "por_Latn": "pt",   # португальский
    "tur_Latn": "tr",   # турецкий
    "ukr_Cyrl": "uk",   # украинский
    "deu_Latn": "de",   # немецкий
}

N_PER_LANG = 700   # берём с запасом, потом дедуплицируем

parts = []
for code, short in flores_codes.items():
    ds = load_dataset("cis-lmu/Glot500", code, split="train", streaming=True)
    texts = []
    for row in islice(iter(ds), N_PER_LANG * 3):   # запас на дедуп
        t = row["text"].strip()
        if 10 <= len(t) <= 400:                    # отсекаем мусор и слишком длинное
            texts.append(t)
        if len(texts) >= N_PER_LANG:
            break
    parts.append(pd.DataFrame({"text": texts, "lang": short}))
    print(f"{short}: {len(texts)} примеров")

lang_df = pd.concat(parts, ignore_index=True).drop_duplicates(subset="text")
print("\nВсего:", len(lang_df))
print(lang_df["lang"].value_counts())

In [ ]:
#2 split 80/20 с стратификацией
from sklearn.model_selection import train_test_split

train_lang, test_lang = train_test_split(
    lang_df, test_size=0.20, stratify=lang_df["lang"], random_state=42
)
print(len(train_lang), len(test_lang))

### Шаг 2. Эксперимент с предварительной обработкой

Сравнить две стратегии предварительной обработки.

**Подход A — Интенсивная предварительная обработка:**

- Привести тексты к нижнему регистру.
- Удалить цифры.
- Удалить знаки препинания и специальные символы.

**Подход B — Минимальная предварительная обработка:**

- Сохранить символы, характерные для отдельных языков.
- Удалить только лишние пробелы и артефакты форматирования.

Объяснить влияние предварительной обработки на распознавание казахского и других языков, содержащих специфические символы.

> **Важно:** оба подхода должны сохранять допустимые буквы Unicode. Не допускается удаление специфических букв казахского языка, таких как ə, ғ, қ, ң, ө, ұ, ү, һ и і.

In [ ]:
import re

def preprocess_A(text):
    # интенсивная: нижний регистр, без цифр, без пунктуации
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    # удаляем знаки препинания, но сохраняем буквы Unicode (включая ə, ғ, қ…)
    text = "".join(c for c in text if c.isalpha() or c.isspace())
    return re.sub(r"\s+", " ", text).strip()

def preprocess_B(text):
    # минимальная: только нормализация пробелов
    return re.sub(r"\s+", " ", text).strip()

for col, fn in [("text_A", preprocess_A), ("text_B", preprocess_B)]:
    train_lang[col] = train_lang["text"].apply(fn)
    test_lang[col]  = test_lang["text"].apply(fn)

# Проверка: специфические буквы казахского сохранились
sample_kk = train_lang[train_lang["lang"] == "kk"].iloc[0]["text_A"]
print("Казахский (preprocess A):", sample_kk[:120])

### Шаг 3. Представление текстовых признаков + Шаг 4. Разработка моделей

**Шаг 3 — признаки:**

1. TF-IDF на уровне слов (Word-level TF-IDF).
2. TF-IDF на уровне символов с использованием символьных n-грамм (1–3).

**Шаг 4 — модели:**

- Multinomial Naive Bayes.
- Logistic Regression.

Сравнить обученные модели с предварительно обученной системой определения языка, например fastText lid.176.

Предварительно обученную модель необходимо оценивать на той же отложенной тестовой выборке.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

y_train = train_lang["lang"].values
y_test  = test_lang["lang"].values

configs = {
    "word_A":  TfidfVectorizer(analyzer="word",      ngram_range=(1, 1), min_df=2),
    "word_B":  TfidfVectorizer(analyzer="word",      ngram_range=(1, 1), min_df=2),
    "char_A":  TfidfVectorizer(analyzer="char_wb",   ngram_range=(1, 3), min_df=2),
    "char_B":  TfidfVectorizer(analyzer="char_wb",   ngram_range=(1, 3), min_df=2),
}

col_for = {"word_A": "text_A", "word_B": "text_B", "char_A": "text_A", "char_B": "text_B"}

rows = []
best = None
for name, vec in configs.items():
    col = col_for[name]
    Xtr = vec.fit_transform(train_lang[col])
    Xte = vec.transform(test_lang[col])
    for mname, model in [("NB", MultinomialNB()), ("LR", LogisticRegression(max_iter=1000))]:
        model.fit(Xtr, y_train)
        pred = model.predict(Xte)
        acc = accuracy_score(y_test, pred)
        f1  = f1_score(y_test, pred, average="macro")
        rows.append({"features": name, "model": mname, "acc": round(acc, 4), "f1": round(f1, 4)})
        if best is None or f1 > best["f1"]:
            best = {"name": name, "model": mname, "f1": f1,
                    "vec": vec, "clf": model, "col": col, "pred": pred}

results = pd.DataFrame(rows)
print(results.sort_values("f1", ascending=False).to_string(index=False))
print(f"\nЛучшая конфигурация: {best['name']} + {best['model']}  (macro-F1={best['f1']:.4f})")

### Сравнение с предобученной моделью fastText lid.176

In [ ]:
#1 загружаем предобученную модель один раз
import os, urllib.request, fasttext

MODEL_PATH = "lid.176.bin"
if not os.path.exists(MODEL_PATH):
    print("Скачиваю fastText lid.176 (~130 MB)...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin",
        MODEL_PATH
    )

ft = fasttext.load_model(MODEL_PATH)

def ft_predict(text):
    label, _ = ft.predict(text.replace("\n", " "))
    return label[0].replace("__label__", "")

ft_pred = test_lang["text"].apply(ft_predict).values
print("fastText acc:",
      round(accuracy_score(y_test, ft_pred), 4),
      "macro-F1:", round(f1_score(y_test, ft_pred, average="macro"), 4))

### Шаг 5. Оценка качества моделей

Рассчитать:

- Accuracy.
- Macro F1-score.
- Confusion Matrix.

Сравнить результаты для различных стратегий предварительной обработки, способов представления признаков и моделей классификации.

Определить наиболее эффективную конфигурацию.

In [ ]:
from sklearn.metrics import confusion_matrix

langs = sorted(lang_df["lang"].unique())
cm = confusion_matrix(y_test, best["pred"], labels=langs)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=langs, yticklabels=langs)
plt.xlabel("predicted"); plt.ylabel("true")
plt.title(f"Confusion matrix — {best['name']} + {best['model']}")
plt.show()

### Шаг 6. Тестирование устойчивости модели

Исследовать поведение модели в трёх условиях.

#### Эксперимент A — Короткие тексты

Сравнить качество определения языка для текстов различной длины:

- 1–3 слова.
- 4–10 слов.
- Более 10 слов.

#### Эксперимент B — Тексты с ошибками

Создать модифицированную версию тестовой выборки, добавив контролируемые орфографические ошибки, например:

- Удаление символов.
- Вставка дополнительных символов.
- Замена символов.

Сравнить качество классификации до и после внесения ошибок.

#### Эксперимент C — Смешение языков (Code-Switching)

- Составить не менее 30 предложений со смешением казахского и русского языков.
- Проанализировать предсказания обученной модели для смешанных текстов.
- Объяснить, почему классификатор с одной языковой меткой может некорректно обрабатывать смешение языков.

In [ ]:
test_eval = test_lang.copy()
test_eval["pred"] = best["pred"]
test_eval["n_words"] = test_eval["text"].str.split().str.len()

def bucket(n):
    if n <= 3:  return "1-3"
    if n <= 10: return "4-10"
    return "10+"

test_eval["bucket"] = test_eval["n_words"].apply(bucket)
for b in ["1-3", "4-10", "10+"]:
    sub = test_eval[test_eval["bucket"] == b]
    if len(sub) == 0:
        continue
    acc = accuracy_score(sub["lang"], sub["pred"])
    print(f"len {b:5s}  n={len(sub):4d}  acc={acc:.4f}")

**B — Тексты с ошибками.**

In [ ]:
import random
rng = random.Random(0)
alphabet = "abcdeiouğıəқңөұүһ"

def noisify(text, p=0.05):
    out = []
    for ch in text:
        r = rng.random()
        if r < p:
            continue                            # удаляем
        elif r < 2 * p:
            out.append(rng.choice(alphabet))    # вставляем лишний символ
            out.append(ch)
        elif r < 3 * p and ch.isalpha():
            out.append(rng.choice(alphabet))    # заменяем
        else:
            out.append(ch)
    return "".join(out)

noisy = test_lang[best["col"]].apply(noisify)
Xn = best["vec"].transform(noisy)
pred_n = best["clf"].predict(Xn)
print("На зашумлённом тесте: acc =",
      round(accuracy_score(y_test, pred_n), 4),
      " macro-F1 =",
      round(f1_score(y_test, pred_n, average='macro'), 4))

**C — Code-switching (kk ⇄ ru).**

In [ ]:
mixed = [
    "Сәлеметсіз бе, как дела?",
    "Мен сағат беске кофе ішемін, потом иду на работу.",
    "Бұл есеп очень сложный болды.",
    "Таңертең жүгірдім, а вечером отдыхал.",
    "Қаладағы движение қатты интенсивный.",
    "Маған кофе қажет, срочно.",
    "Ол концертке барды, но мне не понравилось.",
    "Бұл кітап интересная, прочитал за ночь.",
    "Жаңалықтарды қарадым, ничего нового.",
    "Концерт прошёл сәтті.",
    "Апам пирог жасады, вкусно получилось.",
    "Университетте лекция болды, по математике.",
    "Қалай ойлайсың, стоит ли идти?",
    "Мен бүгін тойдым, не могу больше есть.",
    "Ол машинамен келді, прямо к подъезду.",
    "Жаңа фильм көрдім, супер.",
    "Қазір сөйлесе алмаймын, перезвоню.",
    "Жұмыстан кейін кездесеміз, в кафе.",
    "Ол ағылшынша говорит жақсы.",
    "Бұл есеп тяжёлый, нужно время.",
    "Жаңа үй сатып алдым, ипотека на 15 лет.",
    "Мен бүгін тойға барамын, в семь вечера.",
    "Апам пирог бысырады, с яблоками.",
    "Жаңа жоба басталды, срочный.",
    "Университет жабық, карантин жарияланды.",
    "Концерт жақсы болды, зал был полный.",
    "Жаңа планшет сатып алдым, для работы.",
    "Жаңбыр жауды, промок до нитки.",
    "Мен бүгін ертерек тұрдым, в шесть утра.",
    "Жаңа курс басталды, по программированию.",
]

print(f"Примеров: {len(mixed)}")
pred_mix = [ft_predict(m) for m in mixed]
for m, p in list(zip(mixed, pred_mix))[:10]:
    print(f"[{p}] {m}")

Классификатор возвращает **одну** метку языка, поэтому для смешанного текста результат неустойчив:
обычно побеждает язык с большей долей слов/символов в данном предложении. Это не ошибка модели — это
несоответствие постановки задачи, где у текста по сути два языка одновременно.

### Шаг 7. Анализ ошибок

- Найти не менее пяти неправильных предсказаний.
- Проанализировать влияние длины текста, общей лексики, орфографических ошибок и близкородственных языков.
- Объяснить, почему символьные признаки могут превосходить признаки на уровне слов при обработке коротких многоязычных текстов.
- Предложить способы повышения надёжности систем автоматического определения языка.